In [0]:
CATALOGO = "workspace"
SILVER = f"{CATALOGO}.silver.contratacoes"
GOLD = f"{CATALOGO}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")
spark.sql(f"COMMENT ON SCHEMA {GOLD} IS 'Camada Gold: esquema estrela das contratações públicas do PNCP no Pará, pronto para análise.'")

from pyspark.sql import functions as F, types as T, Window

silver = spark.table(SILVER)

In [0]:
janela_orgao = Window.partitionBy("orgao_cnpj").orderBy(F.col("data_publicacao_pncp").desc_nulls_last())

dim_orgao = (
    silver
    .filter(F.col("orgao_cnpj").isNotNull())
    .withColumn("_rn", F.row_number().over(janela_orgao))
    .filter(F.col("_rn") == 1)
    .select(
        F.col("orgao_cnpj"),
        F.col("orgao_razao_social"),
        F.col("orgao_esfera_id"),
        F.when(F.col("orgao_esfera_id") == "F", "Federal")
         .when(F.col("orgao_esfera_id") == "E", "Estadual")
         .when(F.col("orgao_esfera_id") == "M", "Municipal")
         .when(F.col("orgao_esfera_id") == "D", "Distrital")
         .otherwise("Não informado").alias("orgao_esfera_nome"),
        F.col("orgao_poder_id"),
        F.when(F.col("orgao_poder_id") == "E", "Executivo")
         .when(F.col("orgao_poder_id") == "L", "Legislativo")
         .when(F.col("orgao_poder_id") == "J", "Judiciário")
         .otherwise("Não informado").alias("orgao_poder_nome"),
    )
)

dim_orgao.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.dim_orgao")

In [0]:
dim_municipio = (
    silver
    .filter(F.col("municipio_codigo_ibge").isNotNull())
    .groupBy("municipio_codigo_ibge")
    .agg(
        F.max("municipio_nome").alias("municipio_nome"),
        F.max("uf_sigla").alias("uf_sigla"),
    )
)

dim_municipio.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.dim_municipio")

In [0]:
CONTRATACAO_DIRETA = [8, 9]  # 8 = Dispensa de Licitação, 9 = Inexigibilidade

dim_modalidade = (
    silver
    .filter(F.col("modalidade_id").isNotNull())
    .groupBy("modalidade_id")
    .agg(F.max("modalidade_nome").alias("modalidade_nome"))
    .withColumn("contratacao_direta", F.col("modalidade_id").isin(CONTRATACAO_DIRETA))
    .withColumn(
        "forma_contratacao",
        F.when(F.col("contratacao_direta"), "Contratação direta").otherwise("Licitação"),
    )
)

dim_modalidade.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.dim_modalidade")

In [0]:
dim_tempo = (
    silver
    .filter(F.col("data_publicacao").isNotNull())
    .select(F.col("data_publicacao").alias("data"))
    .distinct()
    .select(
        "data",
        F.year("data").alias("ano"),
        F.month("data").alias("mes"),
        F.date_format("data", "yyyy-MM").alias("ano_mes"),
        F.quarter("data").alias("trimestre"),
        F.dayofmonth("data").alias("dia"),
        F.dayofweek("data").alias("dia_semana_num"),
        F.date_format("data", "EEEE").alias("dia_semana_nome"),
    )
)

dim_tempo.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.dim_tempo")

In [0]:
fato_contratacao = silver.select(
    "numero_controle_pncp",
    "orgao_cnpj",
    "municipio_codigo_ibge",
    "modalidade_id",
    F.col("data_publicacao").alias("data_publicacao"),
    "numero_compra",
    "ano_compra",
    "objeto_compra",
    "situacao_compra_nome",
    "modo_disputa_nome",
    "tipo_instrumento_nome",
    "srp",
    "desfecho",
    "unidade_nome",
    "valor_total_estimado",
    "valor_total_homologado",
    "economia_valor",
    "economia_pct",
    "prazo_proposta_dias",
    "estimado_valido",
    "tem_homologado",
    (F.col("valor_total_homologado") == F.col("valor_total_estimado")).alias("valor_identico"),
)

(
    fato_contratacao.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .clusterBy("modalidade_id", "data_publicacao")
    .saveAsTable(f"{GOLD}.fato_contratacao")
)

In [0]:
COMENTARIOS_TABELA = {
    "dim_orgao": "Dimensão de órgãos contratantes. Uma linha por CNPJ, com a razão social mais recente observada. Origem: silver.contratacoes (campo aninhado orgaoEntidade da API do PNCP).",
    "dim_municipio": "Dimensão de municípios da unidade administrativa responsável pela contratação. Uma linha por código IBGE. Origem: silver.contratacoes (campo aninhado unidadeOrgao).",
    "dim_modalidade": "Dimensão de modalidades de contratação da Lei 14.133/2021, com classificação entre licitação e contratação direta. Origem: silver.contratacoes.",
    "dim_tempo": "Dimensão de calendário derivada das datas de publicação observadas no PNCP. Contém apenas datas com ao menos uma contratação.",
    "fato_contratacao": "Tabela fato das contratações publicadas no PNCP (UF=PA). Granularidade: uma linha por numero_controle_pncp. Métricas de valor e economia; flags indicam limitações de preenchimento da fonte. Origem: silver.contratacoes.",
}

COMENTARIOS_COLUNA = {
    "dim_orgao": {
        "orgao_cnpj": "Chave primária. CNPJ do órgão contratante, 14 dígitos sem formatação. Origem: orgaoEntidade.cnpj.",
        "orgao_razao_social": "Razão social do órgão, com espaços aparados. Origem: orgaoEntidade.razaoSocial.",
        "orgao_esfera_id": "Código da esfera administrativa. Domínio: F (federal), E (estadual), M (municipal), D (distrital).",
        "orgao_esfera_nome": "Descrição da esfera. Derivado de orgao_esfera_id nesta camada.",
        "orgao_poder_id": "Código do poder. Domínio: E (executivo), L (legislativo), J (judiciário).",
        "orgao_poder_nome": "Descrição do poder. Derivado de orgao_poder_id nesta camada.",
    },
    "dim_municipio": {
        "municipio_codigo_ibge": "Chave primária. Código IBGE do município, 7 dígitos. Origem: unidadeOrgao.codigoIbge.",
        "municipio_nome": "Nome do município. Origem: unidadeOrgao.municipioNome.",
        "uf_sigla": "Sigla da unidade federativa. Domínio neste recorte: PA.",
    },
    "dim_modalidade": {
        "modalidade_id": "Chave primária. Código da modalidade no PNCP. Domínio: 1 a 13.",
        "modalidade_nome": "Nome da modalidade. Ex.: Pregão - Eletrônico, Dispensa de Licitação.",
        "contratacao_direta": "Booleano. Verdadeiro para dispensa (8) e inexigibilidade (9). Classificação atribuída nesta camada com base na Lei 14.133/2021.",
        "forma_contratacao": "Rótulo textual: Licitação ou Contratação direta. Derivado de contratacao_direta.",
    },
    "dim_tempo": {
        "data": "Chave primária. Data de publicação no PNCP.",
        "ano": "Ano da data. Domínio neste recorte: 2026.",
        "mes": "Mês da data. Domínio: 1 a 12.",
        "ano_mes": "Ano e mês no formato yyyy-MM, para agrupamentos mensais.",
        "trimestre": "Trimestre da data. Domínio: 1 a 4.",
        "dia": "Dia do mês. Domínio: 1 a 31.",
        "dia_semana_num": "Dia da semana como número. Domínio: 1 (domingo) a 7 (sábado).",
        "dia_semana_nome": "Nome do dia da semana.",
    },
    "fato_contratacao": {
        "numero_controle_pncp": "Chave primária. Identificador único da contratação no PNCP, no formato cnpj-1-sequencial/ano.",
        "orgao_cnpj": "Chave estrangeira para dim_orgao.",
        "municipio_codigo_ibge": "Chave estrangeira para dim_municipio.",
        "modalidade_id": "Chave estrangeira para dim_modalidade.",
        "data_publicacao": "Chave estrangeira para dim_tempo. Data de publicação no PNCP, derivada do timestamp dataPublicacaoPncp.",
        "numero_compra": "Número da compra atribuído pelo órgão. Texto livre, sem padrão garantido.",
        "ano_compra": "Ano da compra informado pelo órgão. Pode divergir do ano de publicação.",
        "objeto_compra": "Descrição textual do objeto contratado, com espaços múltiplos normalizados na Silver.",
        "situacao_compra_nome": "Situação da contratação no PNCP. Domínio observado: Divulgada no PNCP, Suspensa, Revogada, Anulada.",
        "modo_disputa_nome": "Modo de disputa. Ex.: Aberto, Fechado.",
        "tipo_instrumento_nome": "Tipo de instrumento convocatório. Ex.: Edital, Aviso de contratação direta.",
        "srp": "Booleano. Indica se a contratação usa Sistema de Registro de Preços.",
        "desfecho": "Classificação do desfecho atribuída na Silver. Domínio: homologada, frustrada (revogada ou anulada), suspensa, sem_homologacao (sem valor homologado, podendo estar em andamento).",
        "unidade_nome": "Nome da unidade administrativa responsável. Origem: unidadeOrgao.nomeUnidade.",
        "valor_total_estimado": "Valor total estimado da contratação, em reais. Domínio: maior ou igual a zero; zeros ocorrem e indicam não preenchimento.",
        "valor_total_homologado": "Valor total homologado, em reais. Nulo quando a contratação não foi concluída ou o órgão não atualizou o PNCP.",
        "economia_valor": "Diferença entre estimado e homologado, em reais. Nulo quando algum dos valores é ausente ou zero. Calculado na Silver.",
        "economia_pct": "Economia percentual sobre o valor estimado, arredondada em uma casa. Valores negativos indicam contratação acima do estimado. Calculado na Silver.",
        "prazo_proposta_dias": "Dias entre a abertura e o encerramento do prazo de propostas. Calculado na Silver.",
        "estimado_valido": "Booleano. Verdadeiro quando valor_total_estimado é maior que zero.",
        "tem_homologado": "Booleano. Verdadeiro quando valor_total_homologado é maior que zero.",
        "valor_identico": "Booleano. Verdadeiro quando o homologado é exatamente igual ao estimado, o que sugere preenchimento formal do campo e não economia zero. Deve ser filtrado nas análises de economia.",
    },
}

for tabela, comentario in COMENTARIOS_TABELA.items():
    texto = comentario.replace("'", "''")
    spark.sql(f"COMMENT ON TABLE {GOLD}.{tabela} IS '{texto}'")
    for coluna, descricao in COMENTARIOS_COLUNA[tabela].items():
        texto_col = descricao.replace("'", "''")
        spark.sql(f"ALTER TABLE {GOLD}.{tabela} ALTER COLUMN {coluna} COMMENT '{texto_col}'")

print("Catálogo aplicado. Confira em Catalog > workspace > gold.")

Catálogo aplicado. Confira em Catalog > workspace > gold.


In [0]:
RESTRICOES = [
    f"ALTER TABLE {GOLD}.dim_orgao ALTER COLUMN orgao_cnpj SET NOT NULL",
    f"ALTER TABLE {GOLD}.dim_orgao ADD CONSTRAINT pk_dim_orgao PRIMARY KEY (orgao_cnpj)",
    f"ALTER TABLE {GOLD}.dim_municipio ALTER COLUMN municipio_codigo_ibge SET NOT NULL",
    f"ALTER TABLE {GOLD}.dim_municipio ADD CONSTRAINT pk_dim_municipio PRIMARY KEY (municipio_codigo_ibge)",
    f"ALTER TABLE {GOLD}.dim_modalidade ALTER COLUMN modalidade_id SET NOT NULL",
    f"ALTER TABLE {GOLD}.dim_modalidade ADD CONSTRAINT pk_dim_modalidade PRIMARY KEY (modalidade_id)",
    f"ALTER TABLE {GOLD}.dim_tempo ALTER COLUMN data SET NOT NULL",
    f"ALTER TABLE {GOLD}.dim_tempo ADD CONSTRAINT pk_dim_tempo PRIMARY KEY (data)",
    f"ALTER TABLE {GOLD}.fato_contratacao ALTER COLUMN numero_controle_pncp SET NOT NULL",
    f"ALTER TABLE {GOLD}.fato_contratacao ADD CONSTRAINT pk_fato PRIMARY KEY (numero_controle_pncp)",
    f"ALTER TABLE {GOLD}.fato_contratacao ADD CONSTRAINT fk_fato_orgao FOREIGN KEY (orgao_cnpj) REFERENCES {GOLD}.dim_orgao",
    f"ALTER TABLE {GOLD}.fato_contratacao ADD CONSTRAINT fk_fato_municipio FOREIGN KEY (municipio_codigo_ibge) REFERENCES {GOLD}.dim_municipio",
    f"ALTER TABLE {GOLD}.fato_contratacao ADD CONSTRAINT fk_fato_modalidade FOREIGN KEY (modalidade_id) REFERENCES {GOLD}.dim_modalidade",
    f"ALTER TABLE {GOLD}.fato_contratacao ADD CONSTRAINT fk_fato_tempo FOREIGN KEY (data_publicacao) REFERENCES {GOLD}.dim_tempo",
]

for comando in RESTRICOES:
    try:
        spark.sql(comando)
    except Exception as e:
        print(f"[ignorado] {comando.split('ALTER TABLE ')[1][:70]}... -> {str(e)[:120]}")

In [0]:
for tabela in ["fato_contratacao", "dim_orgao", "dim_municipio", "dim_modalidade", "dim_tempo"]:
    print(f"{tabela:<22} {spark.table(f'{GOLD}.{tabela}').count():>8} linhas")

display(spark.sql(f"""
SELECT
  SUM(CASE WHEN o.orgao_cnpj IS NULL THEN 1 ELSE 0 END)             AS orgaos_orfaos,
  SUM(CASE WHEN m.municipio_codigo_ibge IS NULL THEN 1 ELSE 0 END)  AS municipios_orfaos,
  SUM(CASE WHEN md.modalidade_id IS NULL THEN 1 ELSE 0 END)         AS modalidades_orfas,
  SUM(CASE WHEN t.data IS NULL THEN 1 ELSE 0 END)                   AS datas_orfas
FROM {GOLD}.fato_contratacao f
LEFT JOIN {GOLD}.dim_orgao      o  ON f.orgao_cnpj = o.orgao_cnpj
LEFT JOIN {GOLD}.dim_municipio  m  ON f.municipio_codigo_ibge = m.municipio_codigo_ibge
LEFT JOIN {GOLD}.dim_modalidade md ON f.modalidade_id = md.modalidade_id
LEFT JOIN {GOLD}.dim_tempo      t  ON f.data_publicacao = t.data
"""))

fato_contratacao          17432 linhas
dim_orgao                   559 linhas
dim_municipio               144 linhas
dim_modalidade               11 linhas
dim_tempo                   220 linhas


orgaos_orfaos,municipios_orfaos,modalidades_orfas,datas_orfas
0,0,0,0
